# cURL Data

But this file doesn't work any.

In [ ]:
# import requests
# import json
# import os

# url = 'https://vote62-general-66-site.s3.ap-southeast-1.amazonaws.com/structure_f-69-1.json?q=6.20260207.02'
# headers = {
#     'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36',
#     'Referer': 'https://www.vote62.com/'
# }

# response = requests.get(url, headers=headers)

# if response.status_code == 200:
#     data = response.json()
    
#     file_path = './data/result.json'
    
#     os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
#     with open(file_path, 'w', encoding='utf-8') as f:
#         json.dump(data, f, ensure_ascii=False, indent=4)
        
#     print(f"✅ ดึงข้อมูลและเซฟไฟล์ลง '{file_path}' สำเร็จเรียบร้อย!")
    
# else:
#     print(f"❌ ดึงข้อมูลไม่สำเร็จ Status Code: {response.status_code}")

✅ ดึงข้อมูลและเซฟไฟล์ลง './data/result.json' สำเร็จเรียบร้อย!


# Scrape

## List of พรรค

In [ ]:
import json

with open('./data/result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

all_parties = set()

if 'candidates' in data:
    for person in data['candidates']:
        party_name = person.get('party')
        if party_name: 
            all_parties.add(party_name.strip())

PARTIES_CONSTANT = sorted(list(all_parties))

print(f"พบทั้งหมด {len(PARTIES_CONSTANT)} พรรค:")
print(PARTIES_CONSTANT)

with open('constants.py', 'w', encoding='utf-8') as f:
    f.write(f"ALL_PARTIES = {PARTIES_CONSTANT}")

พบทั้งหมด 60 พรรค:
['กรีน', 'กล้าธรรม', 'ก้าวอิสระ', 'ครูไทยเพื่อประชาชน', 'คลองไทย', 'ความหวังใหม่', 'ทางเลือกใหม่', 'ท้องที่ไทย', 'ประชากรไทย', 'ประชาชน', 'ประชาชาติ', 'ประชาธิปัตย์', 'ประชาธิปไตยใหม่', 'ประชาอาสาชาติ', 'ประชาไทย', 'ปวงชนไทย', 'พร้อม', 'พลวัต', 'พลังธรรมใหม่', 'พลังประชารัฐ', 'พลังสังคมใหม่', 'พลังเพื่อไทย', 'พลังไทยรักชาติ', 'ฟิวชัน', 'ภูมิใจไทย', 'มิติใหม่', 'รวมพลัง', 'รวมพลังประชาชน', 'รวมใจไทย', 'รวมไทยสร้างชาติ', 'รักชาติ', 'รักษ์ธรรม', 'วิชชั่นใหม่', 'สร้างอนาคตไทย', 'สังคมประชาธิปไตยไทย', 'สัมมาธิปไตย', 'อนาคตไทย', 'เครือข่ายชาวนาแห่งประเทศไทย', 'เป็นธรรม', 'เพื่อชาติไทย', 'เพื่อชีวิตใหม่', 'เพื่อบ้านเมือง', 'เพื่อไทย', 'เศรษฐกิจ', 'เสรีรวมไทย', 'แผ่นดินธรรม', 'แรงงานสร้างชาติ', 'โอกาสใหม่', 'ใหม่', 'ไทยก้าวหน้า', 'ไทยก้าวใหม่', 'ไทยชนะ', 'ไทยทรัพย์ทวี', 'ไทยธรรม', 'ไทยพร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยภักดี', 'ไทยรวมไทย', 'ไทยสร้างไทย', 'ไทรวมพลัง']


## Runner

In [ ]:
import pandas as pd
import re
import time
import urllib.parse
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from constants import PARTIES_CONSTANT 

driver = webdriver.Chrome() 
scraped_data = []

for party in PARTIES_CONSTANT:
    encoded_party = urllib.parse.quote(party)
    url = f"https://www.vote62.com/69/party/{encoded_party}/"
    driver.get(url)
    
    row = {'Party': party, 'สส_เขต': 0, 'สส_บัญชีรายชื่อ': 0}
    
    try:
        # 1. รอให้ Tag <p> ที่มีคลาส 'font-body text-sm' และมีคำว่า 'สส.' ปรากฏ
        # เราใช้ XPath ที่เจาะจงคลาสตามที่คุณแคปมาเลย
        wait = WebDriverWait(driver, 10)
        element = wait.until(EC.presence_of_element_located(
            (By.XPATH, "//p[contains(@class, 'font-body') and contains(., 'สส.')]")
        ))
        
        # 2. ดึง text ออกมาทั้งหมด (มันจะได้: "สส. เขต 2 คน\nสส. บัญชีรายชื่อ 12 คน")
        full_text = element.text
        # แทนที่ \n ด้วยช่องว่างก่อน
        cleaned_text = full_text.replace('\n', ' ')
        
        # แล้วค่อยปริ้นท์
        print(f"✅ ดึงข้อมูล {party} สำเร็จ: {cleaned_text}")

        # 3. ใช้ Regex ดึงตัวเลขที่อยู่หลังคำที่ต้องการ
        # หาตัวเลขที่อยู่หลัง 'สส. เขต'
        district_match = re.search(r'สส\.\s*เขต\s*(\d+)', full_text)
        if district_match:
            row['สส_เขต'] = int(district_match.group(1))
            
        # หาตัวเลขที่อยู่หลัง 'บัญชีรายชื่อ'
        list_match = re.search(r'บัญชีรายชื่อ\s*(\d+)', full_text)
        if list_match:
            row['สส_บัญชีรายชื่อ'] = int(list_match.group(1))
            
    except Exception as e:
        print(f"⚠️ พรรค {party} หาข้อมูลไม่เจอ")

    scraped_data.append(row)
    time.sleep(1)

driver.quit()

# บันทึกเป็น CSV
df = pd.DataFrame(scraped_data)
df.to_csv('./data/vote62_party_stats.csv', index=False, encoding='utf-8-sig')
print("✅ บันทึกไฟล์สำเร็จ!")

✅ ดึงข้อมูล กรีน สำเร็จ: สส. เขต 2 คน สส. บัญชีรายชื่อ 12 คน
✅ ดึงข้อมูล กล้าธรรม สำเร็จ: สส. เขต 334 คน สส. บัญชีรายชื่อ 100 คน
✅ ดึงข้อมูล ก้าวอิสระ สำเร็จ: สส. เขต 6 คน สส. บัญชีรายชื่อ 15 คน
✅ ดึงข้อมูล ครูไทยเพื่อประชาชน สำเร็จ: สส. เขต 1 คน สส. บัญชีรายชื่อ 22 คน
✅ ดึงข้อมูล คลองไทย สำเร็จ: สส. เขต 2 คน สส. บัญชีรายชื่อ 16 คน
✅ ดึงข้อมูล ความหวังใหม่ สำเร็จ: สส. เขต 1 คน สส. บัญชีรายชื่อ 8 คน
✅ ดึงข้อมูล ทางเลือกใหม่ สำเร็จ: สส. เขต 43 คน สส. บัญชีรายชื่อ 20 คน
✅ ดึงข้อมูล ท้องที่ไทย สำเร็จ: สส. เขต 2 คน สส. บัญชีรายชื่อ 8 คน
✅ ดึงข้อมูล ประชากรไทย สำเร็จ: สส. เขต 42 คน สส. บัญชีรายชื่อ 18 คน
✅ ดึงข้อมูล ประชาชน สำเร็จ: สส. เขต 400 คน สส. บัญชีรายชื่อ 99 คน
✅ ดึงข้อมูล ประชาชาติ สำเร็จ: สส. เขต 15 คน สส. บัญชีรายชื่อ 37 คน
✅ ดึงข้อมูล ประชาธิปัตย์ สำเร็จ: สส. เขต 400 คน สส. บัญชีรายชื่อ 98 คน
✅ ดึงข้อมูล ประชาธิปไตยใหม่ สำเร็จ: สส. เขต 70 คน สส. บัญชีรายชื่อ 21 คน
✅ ดึงข้อมูล ประชาอาสาชาติ สำเร็จ: สส. เขต 1 คน สส. บัญชีรายชื่อ 3 คน
✅ ดึงข้อมูล ประชาไทย สำเร็จ: สส. เขต 1 คน สส. บั